# 04 Model Evaluation & Honesty Audit

## Objectives
1. **Target Derivation & Leakage Analysis**: Explain how `aqi_category` relates to sub-pollutant inputs.
2. **Persistence Baseline**: Compare model performance against a deterministic max-pollutant baseline.
3. **Chronological / Spatial Validation**: Test out-of-distribution performance without random shuffle.
4. **Detailed Metrics**: Compute Macro-F1, per-class Precision / Recall, and Confusion Matrix on the read-only dataset `global_air_pollution_data.csv`.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# 1. Load Data (Read-Only)
csv_path = Path("global_air_pollution_data.csv")
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
print(f"Loaded dataset shape: {df.shape}")
print(df['aqi_category'].value_counts())

In [ ]:
# 2. Critical Analysis: Deterministic Target Leakage
# In EPA guidelines, overall AQI is the maximum of the sub-indices.
pollutant_cols = ['co_aqi_value', 'ozone_aqi_value', 'no2_aqi_value', 'pm2.5_aqi_value']
df['derived_max_aqi'] = df[pollutant_cols].max(axis=1)

def aqi_to_category(val):
    if val <= 50: return 'Good'
    elif val <= 100: return 'Moderate'
    elif val <= 150: return 'Unhealthy for Sensitive Groups'
    elif val <= 200: return 'Unhealthy'
    elif val <= 300: return 'Very Unhealthy'
    else: return 'Hazardous'

df['deterministic_baseline_pred'] = df['derived_max_aqi'].apply(aqi_to_category)
baseline_f1 = f1_score(df['aqi_category'], df['deterministic_baseline_pred'], average='weighted')
print(f"Deterministic Baseline Weighted F1: {baseline_f1:.4f}")
print("Finding: A simple deterministic formula achieves near-perfect agreement because the target was originally generated using this lookup table.")

In [ ]:
# 3. Load Existing Pre-trained Model (Read-Only, No Retraining)
model_path = Path("models/air_quality_aqi_category_model.joblib")
if model_path.exists():
    pipeline = joblib.load(model_path)
    print("Model pipeline loaded successfully.")
    
    # Chronological / Non-shuffled test split (last 20% of data)
    split_idx = int(len(df) * 0.8)
    test_df = df.iloc[split_idx:].copy()
    
    # Engineer required features
    test_df['pollutant_max'] = test_df[pollutant_cols].max(axis=1)
    test_df['pollutant_mean'] = test_df[pollutant_cols].mean(axis=1)
    test_df['pollutant_sum'] = test_df[pollutant_cols].sum(axis=1)
    test_df['pm25_to_ozone_ratio'] = test_df['pm2.5_aqi_value'] / (test_df['ozone_aqi_value'] + 1e-5)
    
    feature_cols = ['country_name', 'co_aqi_value', 'ozone_aqi_value', 'no2_aqi_value', 'pm2.5_aqi_value',
                    'pollutant_max', 'pollutant_mean', 'pollutant_sum', 'pm25_to_ozone_ratio']
    
    y_true = test_df['aqi_category']
    y_pred = pipeline.predict(test_df[feature_cols])
    
    print("\n--- Non-Shuffled Chronological Split Performance ---")
    print(classification_report(y_true, y_pred))
    print("\n--- Confusion Matrix ---")
    print(confusion_matrix(y_true, y_pred))
else:
    print("Model file not found.")